# <font color='gold'>Sampling University review text</font>


In [1]:
import pandas as pd
import numpy as np 
from nltk import word_tokenize
from sklearn.model_selection import StratifiedShuffleSplit
from sampling_util.table_util import display_count_table, compare_source_and_sample  # helper functions to make tables

#===========================================================
# Parameters
#===========================================================
random_state = 73

input_file = '../../data/1_processed/3_UExeter_uni_reviews.csv' 

target_sample_size = 500

rating_bins = [1, 2.5, 4, 5]
rating_labels = {'(0.999, 2.5]':'Neg', '(2.5, 4.0]':'Conf/Neu',  '(4.0, 5.0]':'Pos'}

length_bin_interval_size = 20

output_cols = ['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin']
#===========================================================

updated_sourcefile = input_file.replace('.csv', '_with_sampling_criteria.csv')
output_file_name = input_file.replace('1_processed/', '2_sampled/sampling_output_unlabelled/').replace('.csv', f'(N={target_sample_size}).csv')

rating_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_rating_stats.csv')
length_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_length_stats.csv')

print(f'\nInput:      {input_file}\n\n'
      f'Source_v2:  \033[032m{updated_sourcefile}\033[0m\n\n'
      f'Output:     \033[033m{output_file_name}\033[0m\n\n'
      f'Stats_1:    \033[35m{rating_stats_file}\033[0m\n'
      f'Stats_2:    \033[35m{length_stats_file}\033[0m\n')



Input:      ../../data/1_processed/3_UExeter_uni_reviews.csv

Source_v2:  ../../data/1_processed/3_UExeter_uni_reviews_with_sampling_criteria.csv

Output:     ../../data/2_sampled/sampling_output_unlabelled/3_UExeter_uni_reviews(N=500).csv

Stats_1:    ../../data/2_sampled/sampling_output_stats/3_UExeter_uni_reviews(N=500)_rating_stats.csv
Stats_2:    ../../data/2_sampled/sampling_output_stats/3_UExeter_uni_reviews(N=500)_length_stats.csv



## <font color='cornflowerblue'>1) Prepare helper columns</font>


In [2]:
source_df = pd.read_csv(input_file)
source_df = source_df.rename(columns={'rating_overall':'rating'})
print(source_df.shape)

# Get review length (as number of tokens)
source_df.loc[:, 'token_list'] = source_df.review_text.str.strip().apply(word_tokenize)
source_df.loc[:, 'num_token'] = source_df.token_list.apply(len)

# Get sampling_feature distribution
display_count_table(source_df, 'num_token', sort_by='num_token')
display_count_table(source_df, 'rating', sort_by='rating', ascending=True)

print(f"\n\033[34m{'='*50}\033[0m\n") #===========================================

# Bin ratings into groups
source_df.loc[:, 'rating_bin'] = pd.cut(source_df.rating, bins=rating_bins, labels=None, include_lowest=True).astype(str) #, labels=None will show the bin value range
source_df.loc[:, 'rating_cat'] = source_df.rating_bin.map(rating_labels)  #or source_df.rating.apply(lambda x: 'Pos' if 4 < x <= 5 else 'Neg' if 1 <= x <= 2.5 else 'Conf/Neu')
display_count_table(source_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True)
print()

# Bin num_token with the given interval size
interval_size = length_bin_interval_size
maxval = max(source_df.num_token.unique())
bin_edges = np.arange(1, maxval + interval_size, interval_size)

source_df.loc[:, 'length_bin'] = pd.cut(source_df.num_token, bins=bin_edges, labels=None, include_lowest=True)
display_count_table(source_df, 'length_bin', sort_by='length_bin', ascending=True)
print()

#=================================================================
# Create combined column for multi-criteria stratified sampling
source_df.loc[:, 'sample_criteria'] = source_df.rating_cat.astype(str) + '_' + source_df.length_bin.astype(str) 

display_count_table(source_df, 'sample_criteria', sort_by='sample_criteria')

source_df.to_csv(updated_sourcefile, index=False)
print(f'\nsource_df written to \033[33m{updated_sourcefile}\033[0m, done~!\n')
#=================================================================

source_df

(557, 13)


,num_token,count
77,1,1
83,3,1
80,4,1
12,5,14
0,6,29
...,...,...
74,183,1
75,187,1
69,197,1
62,215,1


,rating,count
15,1.0,4
19,1.6,1
23,2.0,1
22,2.2,1
17,2.4,2
21,2.5,1
20,2.6,1
12,2.8,6
11,3.0,8
8,3.2,15


,rating_cat,rating_bin,count
2,Neg,"(0.999, 2.5]",10
1,Conf/Neu,"(2.5, 4.0]",141
0,Pos,"(4.0, 5.0]",406


,length_bin,count
0,"(0.999, 21.0]",331
1,"(21.0, 41.0]",123
2,"(41.0, 61.0]",55
3,"(61.0, 81.0]",15
4,"(81.0, 101.0]",9
6,"(101.0, 121.0]",7
5,"(121.0, 141.0]",9
8,"(141.0, 161.0]",2
10,"(161.0, 181.0]",1
7,"(181.0, 201.0]",3


,sample_criteria,count
2,"Conf/Neu_(0.999, 21.0]",71
15,"Conf/Neu_(101.0, 121.0]",2
17,"Conf/Neu_(121.0, 141.0]",2
16,"Conf/Neu_(141.0, 161.0]",2
23,"Conf/Neu_(201.0, 221.0]",1
4,"Conf/Neu_(21.0, 41.0]",33
5,"Conf/Neu_(41.0, 61.0]",20
8,"Conf/Neu_(61.0, 81.0]",6
12,"Conf/Neu_(81.0, 101.0]",4
11,"Neg_(0.999, 21.0]",4



source_df written to ../../data/1_processed/3_UExeter_uni_reviews_with_sampling_criteria.csv, done~!



,Unnamed: 0,id,index,student_id,date,review_text,rating,rating_facilities,rating_clubs,rating_careerService,rating_internet,source,type,token_list,num_token,rating_bin,rating_cat,length_bin,sample_criteria
0,0,3_1_469_129,0,887111.0,1st June 23,I like my university very much!,5.0,5,5,5,5,data3_557,university review,"[I, like, my, university, very, much, !]",7,"(4.0, 5.0]",Pos,"(0.999, 21.0]","Pos_(0.999, 21.0]"
1,1,3_2_161_233,1,1287025.0,31st May 23,"I am from the USA, so I am comparing my experi...",3.4,3,3,3,5,data3_557,university review,"[I, am, from, the, USA, ,, so, I, am, comparin...",217,"(2.5, 4.0]",Conf/Neu,"(201.0, 221.0]","Conf/Neu_(201.0, 221.0]"
2,2,3_3_160_142,2,1286468.0,20th May 23,Exeter University is generally a lovely place ...,4.0,5,4,4,3,data3_557,university review,"[Exeter, University, is, generally, a, lovely,...",43,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]","Conf/Neu_(41.0, 61.0]"
3,3,3_4_273_142,3,486467.0,20th May 23,Nice facilities and some great lecturers! The ...,4.4,4,4,4,5,data3_557,university review,"[Nice, facilities, and, some, great, lecturers...",57,"(4.0, 5.0]",Pos,"(41.0, 61.0]","Pos_(41.0, 61.0]"
4,4,3_5_272_119,4,486456.0,19th May 23,The uni is good they provide me with alot of s...,4.8,5,5,5,5,data3_557,university review,"[The, uni, is, good, they, provide, me, with, ...",29,"(4.0, 5.0]",Pos,"(21.0, 41.0]","Pos_(21.0, 41.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,552,3_553_1_182,552,NaN,25th March 15,Brilliant university; beautiful campus; active...,4.7,5,5,5,5,data3_557,university review,"[Brilliant, university, ;, beautiful, campus, ...",29,"(4.0, 5.0]",Pos,"(21.0, 41.0]","Pos_(21.0, 41.0]"
553,553,3_554_1_182,553,NaN,25th March 15,Sport membership is very expensive,4.3,4,5,5,5,data3_557,university review,"[Sport, membership, is, very, expensive]",5,"(4.0, 5.0]",Pos,"(0.999, 21.0]","Pos_(0.999, 21.0]"
554,554,3_555_1_182,554,NaN,25th March 15,It is great! But there should be a swimming po...,4.3,4,5,5,5,data3_557,university review,"[It, is, great, !, But, there, should, be, a, ...",14,"(4.0, 5.0]",Pos,"(0.999, 21.0]","Pos_(0.999, 21.0]"
555,555,3_556_1_182,555,NaN,25th March 15,ITS AMAZING! I love my university and wouldn't...,4.7,5,5,5,5,data3_557,university review,"[ITS, AMAZING, !, I, love, my, university, and...",15,"(4.0, 5.0]",Pos,"(0.999, 21.0]","Pos_(0.999, 21.0]"


In [3]:
# ==================================================================================
# As sklearn requires bin with count > 1, 
# identify bins with fewer than 2 samples, duplicate that row (to be removed later)
# ==================================================================================
criteria_bin_counts = source_df['sample_criteria'].value_counts() 
big_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts == max(criteria_bin_counts)].index[0]
single_count_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts < 2].index
print("max_count_bin criteria vals:       ", big_bin_criteria_vals)
print("single_count_bin criteria vals::   ", single_count_bin_criteria_vals, '\n')

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False) 
print(source_df.shape)

# For bins with only 1 sample, duplicate them so that the sampling module won't throw error. 
# Will drop duplicates after sampling if they exist in the selected samples
for criteria_value in single_count_bin_criteria_vals: 
    source_df = pd.concat([source_df, source_df.loc[source_df.sample_criteria == criteria_value]], ignore_index=True, sort=False)

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False)
print(source_df.shape)

max_count_bin criteria vals:        Pos_(0.999, 21.0]
single_count_bin criteria vals::    Index(['Neg_(101.0, 121.0]', 'Pos_(201.0, 221.0]', 'Pos_(161.0, 181.0]',
       'Conf/Neu_(201.0, 221.0]', 'Neg_(81.0, 101.0]'],
      dtype='object', name='sample_criteria') 



,sample_criteria,count
0,"Pos_(0.999, 21.0]",256
1,"Pos_(21.0, 41.0]",88
2,"Conf/Neu_(0.999, 21.0]",71
3,"Pos_(41.0, 61.0]",33
4,"Conf/Neu_(21.0, 41.0]",33
5,"Conf/Neu_(41.0, 61.0]",20
6,"Pos_(61.0, 81.0]",9
7,"Pos_(121.0, 141.0]",7
8,"Conf/Neu_(61.0, 81.0]",6
11,"Neg_(0.999, 21.0]",4


(557, 19)


,sample_criteria,count
0,"Pos_(0.999, 21.0]",256
1,"Pos_(21.0, 41.0]",88
2,"Conf/Neu_(0.999, 21.0]",71
3,"Pos_(41.0, 61.0]",33
4,"Conf/Neu_(21.0, 41.0]",33
5,"Conf/Neu_(41.0, 61.0]",20
6,"Pos_(61.0, 81.0]",9
7,"Pos_(121.0, 141.0]",7
8,"Conf/Neu_(61.0, 81.0]",6
11,"Neg_(0.999, 21.0]",4


(562, 19)


## <font color='cornflowerblue'>2) Extract samples & post-processing</font>


In [4]:
# Get the stratified sample
sss = StratifiedShuffleSplit(n_splits=1, test_size=target_sample_size + len(single_count_bin_criteria_vals), random_state=random_state)
_train_idx, sample_idx = next(sss.split(source_df.copy(), source_df.sample_criteria))  # iterate through n_split, which is 1. Only need the 'sample_idx' variable
 
sample_df = source_df.iloc[sample_idx]
sample_df = sample_df[output_cols]

# now drop the duplicated row added earlier
print("before drop_duplicates:       ", sample_df.shape)
sample_df = sample_df.drop_duplicates(keep='first')
print("after  drop_duplicates:       ", sample_df.shape, '\n')

display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
print()

# If current sample size > target_sample_size, adjust the sample size 
diff = len(sample_df) - target_sample_size
if diff > 0:  
    # diff > 0 means no duplicated rows were selected in this sample, so the drop_duplicates above made no change
    # In this case, drop same-number rows randomly from the largest sample_criteria category
    print(f"\033[35m\nNo duplicated rows in sample.\n\033[36mRemoving {diff} random-sample of sample_criteria == ({big_bin_criteria_vals}) to match target sample size !!!!!\n\033[0m")
    drop_scope = sample_df.loc[sample_df.sample_criteria == big_bin_criteria_vals]
    drop_row_idx = drop_scope.sample(n=diff).index
    sample_df = sample_df.drop(index=drop_row_idx)
    print(sample_df.shape, '\n')
    display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
    print()
else:
    print(f"\033[36m\nDuplicated rows in sample already dropped, all good.\n\033[0m")
    
source_df = source_df.drop(columns=['token_list']).drop_duplicates(keep='first')

sample_df

before drop_duplicates:        (505, 10)
after  drop_duplicates:        (500, 10) 



,sample_criteria,count
0,"Pos_(0.999, 21.0]",230
1,"Pos_(21.0, 41.0]",79
2,"Conf/Neu_(0.999, 21.0]",64
3,"Pos_(41.0, 61.0]",30
4,"Conf/Neu_(21.0, 41.0]",30
5,"Conf/Neu_(41.0, 61.0]",18
6,"Pos_(61.0, 81.0]",8
7,"Pos_(121.0, 141.0]",6
8,"Conf/Neu_(61.0, 81.0]",5
11,"Pos_(81.0, 101.0]",3




Duplicated rows in sample already dropped, all good.



,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
151,university review,data3_557,3_152_29_293,5.0,6,"Pos_(0.999, 21.0]",Really good university to be in,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
502,university review,data3_557,3_503_23_219,4.0,50,"Conf/Neu_(41.0, 61.0]",Exeter is an incredible experience of communit...,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]"
158,university review,data3_557,3_159_459_293,5.0,12,"Pos_(0.999, 21.0]","Great Uni, loved my three years there. Would r...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
242,university review,data3_557,3_243_189_62,5.0,106,"Pos_(101.0, 121.0]",University of Exeter has met expectations I di...,"(4.0, 5.0]",Pos,"(101.0, 121.0]"
280,university review,data3_557,3_281_304_111,5.0,8,"Pos_(0.999, 21.0]","Nice services, good equipments, nice environment","(4.0, 5.0]",Pos,"(0.999, 21.0]"
...,...,...,...,...,...,...,...,...,...,...
342,university review,data3_557,3_343_393_290,4.8,15,"Pos_(0.999, 21.0]","University is absolutely as good as expected, ...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
117,university review,data3_557,3_118_412_64,5.0,25,"Pos_(21.0, 41.0]","Brilliant University, perfect fit for me even ...","(4.0, 5.0]",Pos,"(21.0, 41.0]"
333,university review,data3_557,3_334_178_52,4.2,12,"Pos_(0.999, 21.0]",Great uni! Loads of food outlets and study are...,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
426,university review,data3_557,3_427_169_213,4.6,16,"Pos_(0.999, 21.0]","The campus is amazing, people are all fantasti...","(4.0, 5.0]",Pos,"(0.999, 21.0]"


## <font color='cornflowerblue'>3) Compare source vs. sample distributions</font>


In [5]:
rating_stats = compare_source_and_sample(source_df, sample_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True).fillna('')
rating_stats.to_csv(rating_stats_file, index=False)
print(f'\nRating_stats written to \033[33m{rating_stats_file}\033[0m, done~!\n')

rating_stats


Rating_stats written to ../../data/2_sampled/sampling_output_stats/3_UExeter_uni_reviews(N=500)_rating_stats.csv, done~!



,df_name,rating_cat,rating_bin,count,%,df_name,rating_cat,rating_bin,count,%
2,source_df,Neg,"(0.999, 2.5]",10.0,0.017953,sample_df,Neg,"(0.999, 2.5]",9.0,0.018
1,source_df,Conf/Neu,"(2.5, 4.0]",141.0,0.253142,sample_df,Conf/Neu,"(2.5, 4.0]",127.0,0.254
0,source_df,Pos,"(4.0, 5.0]",406.0,0.728905,sample_df,Pos,"(4.0, 5.0]",364.0,0.728
3,,,,557.0,1.000000,,,,500.0,1.000


In [6]:
length_stats = compare_source_and_sample(source_df, sample_df, ['length_bin'], sort_by='length_bin', ascending=True).fillna('')
length_stats.to_csv(length_stats_file, index=False)
print(f'\nLength_stats written to \033[33m{length_stats_file}\033[0m, done~!\n')

length_stats


Length_stats written to ../../data/2_sampled/sampling_output_stats/3_UExeter_uni_reviews(N=500)_length_stats.csv, done~!



,df_name,length_bin,count,%,df_name,length_bin,count,%
0,source_df,"(0.999, 21.0]",331.0,0.594255,sample_df,"(0.999, 21.0]",297.0,0.594
1,source_df,"(21.0, 41.0]",123.0,0.220826,sample_df,"(21.0, 41.0]",111.0,0.222
2,source_df,"(41.0, 61.0]",55.0,0.098743,sample_df,"(41.0, 61.0]",50.0,0.100
3,source_df,"(61.0, 81.0]",15.0,0.026930,sample_df,"(61.0, 81.0]",13.0,0.026
4,source_df,"(81.0, 101.0]",9.0,0.016158,sample_df,"(121.0, 141.0]",8.0,0.016
6,source_df,"(101.0, 121.0]",7.0,0.012567,sample_df,"(101.0, 121.0]",6.0,0.012
5,source_df,"(121.0, 141.0]",9.0,0.016158,sample_df,"(81.0, 101.0]",7.0,0.014
8,source_df,"(141.0, 161.0]",2.0,0.003591,sample_df,"(141.0, 161.0]",2.0,0.004
10,source_df,"(161.0, 181.0]",1.0,0.001795,sample_df,"(161.0, 181.0]",1.0,0.002
7,source_df,"(181.0, 201.0]",3.0,0.005386,sample_df,"(181.0, 201.0]",3.0,0.006


## <font color='cornflowerblue'>4) Write to file</font>


In [7]:
print(sample_df.shape)
print(list(sample_df.columns.values), '\n')

sample_df.to_csv(output_file_name, index=False)
print(f'sample written to \033[33m{output_file_name}\033[0m, done~!\n')

sample_df

(500, 10)
['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin'] 

sample written to ../../data/2_sampled/sampling_output_unlabelled/3_UExeter_uni_reviews(N=500).csv, done~!



,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
151,university review,data3_557,3_152_29_293,5.0,6,"Pos_(0.999, 21.0]",Really good university to be in,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
502,university review,data3_557,3_503_23_219,4.0,50,"Conf/Neu_(41.0, 61.0]",Exeter is an incredible experience of communit...,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]"
158,university review,data3_557,3_159_459_293,5.0,12,"Pos_(0.999, 21.0]","Great Uni, loved my three years there. Would r...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
242,university review,data3_557,3_243_189_62,5.0,106,"Pos_(101.0, 121.0]",University of Exeter has met expectations I di...,"(4.0, 5.0]",Pos,"(101.0, 121.0]"
280,university review,data3_557,3_281_304_111,5.0,8,"Pos_(0.999, 21.0]","Nice services, good equipments, nice environment","(4.0, 5.0]",Pos,"(0.999, 21.0]"
...,...,...,...,...,...,...,...,...,...,...
342,university review,data3_557,3_343_393_290,4.8,15,"Pos_(0.999, 21.0]","University is absolutely as good as expected, ...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
117,university review,data3_557,3_118_412_64,5.0,25,"Pos_(21.0, 41.0]","Brilliant University, perfect fit for me even ...","(4.0, 5.0]",Pos,"(21.0, 41.0]"
333,university review,data3_557,3_334_178_52,4.2,12,"Pos_(0.999, 21.0]",Great uni! Loads of food outlets and study are...,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
426,university review,data3_557,3_427_169_213,4.6,16,"Pos_(0.999, 21.0]","The campus is amazing, people are all fantasti...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
